## Roblox Discovery Analysis

- Starting Hypothesis:
  - Players struggle to find new, high-quality experiences
  because discovery surfaces favor games that are already popular.


In [1]:
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

files = sorted(glob.glob("../data/raw/games_*.csv"))
raw = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
print(len(files), "pulls,", raw.shape)
raw.head()

3 pulls, (2650, 13)


,universe_id,rootPlaceId,name,genre,playing,visits,favoritedCount,maxPlayers,created,updated,upVotes,downVotes,pulled_at_utc
0,6331902150,18687417158,[UPD 🤖] Forsaken,All,80939,5844951987,3259881,9,2024-07-28T02:20:42.22Z,2026-10-17T17:46:23.495Z,2405873,443359,2026-10-18T21:19:23.660898+00:00
1,5203828273,15101393044,[🎃] Dress To Impress,All,146282,11184301381,7628287,13,2023-10-18T09:11:36.563Z,2026-10-18T21:09:23.484Z,4040475,420625,2026-10-18T21:19:23.660898+00:00
2,10765288803,114326934417838,Break and Steal an Egg,All,80856,176786084,539610,7,2026-09-05T18:30:02.346Z,2026-10-18T19:11:20.205Z,327447,8136,2026-10-18T21:19:23.660898+00:00
3,3647333358,9872472334,[🎃HLWN] Evade,Horror,51335,9333350378,8476821,15,2022-06-10T19:20:37.097Z,2026-10-18T18:34:42.597Z,3261610,212055,2026-10-18T21:19:23.660898+00:00
4,10767386157,103363457515322,🪵Chop A Forest,All,37868,14608760,53758,18,2026-09-21T12:02:18.72Z,2026-10-18T21:18:20.548Z,19086,1539,2026-10-18T21:19:23.660898+00:00


In [2]:
latest = raw.sort_values("pulled_at_utc").groupby("universe_id").last()
avg_playing = raw.groupby("universe_id")["playing"].mean().rename("avg_playing")
pulls_seen = raw.groupby("universe_id").size().rename("pulls_seen")
games = latest.join(avg_playing).join(pulls_seen).reset_index()
print(games.shape)

(942, 15)


In [3]:
games["created"] = pd.to_datetime(games["created"], utc=True, format="ISO8601")
games["updated"] = pd.to_datetime(games["updated"], utc=True, format="ISO8601")
pull_date = pd.to_datetime(raw["pulled_at_utc"], utc=True, format="ISO8601").max()

games["age_days"] = (pull_date - games["created"]).dt.days
games["days_since_update"] = (pull_date - games["updated"]).dt.days
games["total_votes"] = games["upVotes"] + games["downVotes"]
games["like_ratio"] = games["upVotes"] / games["total_votes"]
games["age_group"] = pd.cut(
    games["age_days"],
    bins=[-1, 182, 365, 730, np.inf],
    labels=["<6 mo", "6–12 mo", "1–2 yrs", "2+ yrs"],
)
games[["name", "avg_playing", "age_days", "like_ratio", "age_group"]].head(10).round({"avg_playing": 0, "like_ratio": 3})

,name,avg_playing,age_days,like_ratio,age_group
0,🍕Work at a Pizza Place,3719.0,6779,0.929,2+ yrs
1,Natural Disaster Survival,8306.0,6779,0.908,2+ yrs
2,🌳 Lumber Tycoon 2,3248.0,6290,0.877,2+ yrs
3,🎃 Notoriety: A PAYDAY® Experience,2278.0,6107,0.925,2+ yrs
4,Pilot Training Flight Simulator,3515.0,6132,0.933,2+ yrs
5,Murder Mystery 2,502402.0,4657,0.908,2+ yrs
6,Prison Life,6739.0,4554,0.850,2+ yrs
7,🔨Plane Crazy🔨 [UPDATE],4408.0,4478,0.802,2+ yrs
8,[🎃] Speed Run 4,814.0,4382,0.738,2+ yrs
9,[🌻GREEN HILL] Survive The Disasters 2,25.0,4402,0.887,2+ yrs


In [4]:
print(games["genre"].value_counts())

genre
All              746
Adventure         55
Fighting          34
Town and City     28
Horror            21
RPG               13
Sports            10
Military          10
Building           9
FPS                7
Comedy             6
Sci-Fi             1
Medieval           1
Naval              1
Name: count, dtype: int64


In [5]:
pl_all = pd.concat([pd.read_csv(f) for f in glob.glob("../data/raw/placements_*.csv")])

genre_rows = pl_all[pl_all["sort_name"].str.startswith("Trending in", na=False)].copy()
genre_rows["genre_label"] = genre_rows["sort_name"].str.replace("Trending in ", "", regex=False)

best = (genre_rows.sort_values("position")
        .drop_duplicates("universe_id")[["universe_id", "genre_label"]])

games = games.merge(best, on="universe_id", how="left")
games["genre_label"] = games["genre_label"].fillna("Unlabeled")
print(games["genre_label"].value_counts())

games.to_csv("../data/clean/games.csv", index=False)

genre_label
Unlabeled                445
Simulation                81
Survival                  38
Action                    36
Adventure                 34
Shopping                  33
Obby & Platformer         32
Strategy                  32
Party & Casual            31
Sports & Racing           31
RPG                       31
Roleplay & Avatar Sim     30
Entertainment             30
Shooter                   29
Puzzle                    29
Name: count, dtype: int64


In [6]:
games.groupby(games["genre_label"] == "Unlabeled")["avg_playing"].median()

genre_label
False    3839.666667
True      341.666667
Name: avg_playing, dtype: float64